In [12]:

import re
from typing import List, Tuple, Optional

Fact = Tuple[str, str, str]

factList: List[Fact] = [
    ("Watson", "met", "Holmes"),
    ("Emmanuel", "meet", "DANTE"),
]

def normalize_entity(s: str) -> str:
    return s.strip().lower()

def normalize_verb(v: str) -> str:
    v = v.strip().lower()

    if v in {"met", "meet", "meets", "meeting"}:
        return "meet"
    return v

def normalize_fact(f: Fact) -> Fact:
    subj, verb, obj = f
    return (normalize_entity(subj), normalize_verb(verb), normalize_entity(obj))

normalized_facts = [normalize_fact(f) for f in factList]

WHO_DID_PATTERN = re.compile(r"^\s*who\s+did\s+(.+?)\s+meet\s*\?\s*$", re.IGNORECASE)

def answer(question: str) -> Optional[List[str]]:
    m = WHO_DID_PATTERN.match(question)
    if not m:
        return None

    subject = normalize_entity(m.group(1))

    results = [obj for (subj, verb, obj) in normalized_facts
               if subj == subject and verb == "meet"]

    return results

q = "Who did Watson meet?"
print(answer(q))

['holmes']


In [13]:
factList: List[Fact] = [
    ('Watson', 'met', 'Holmes','WHO_IS','the private detective'),
    ('Emmanuel', 'meet', 'DANTE','WHO_IS','cat'),
]


In [14]:
import sys
import spacy

nlp = spacy.load("en_core_web_sm")

for obj in nlp.pipeline:
    print(obj)

('tok2vec', <spacy.pipeline.tok2vec.Tok2Vec object at 0x000002486A1CD160>)
('tagger', <spacy.pipeline.tagger.Tagger object at 0x000002486A1CDA00>)
('parser', <spacy.pipeline.dep_parser.DependencyParser object at 0x000002486A5260B0>)
('attribute_ruler', <spacy.pipeline.attributeruler.AttributeRuler object at 0x0000024869A9B500>)
('lemmatizer', <spacy.lang.en.lemmatizer.EnglishLemmatizer object at 0x0000024869C77F00>)
('ner', <spacy.pipeline.ner.EntityRecognizer object at 0x000002486A1EC4A0>)


In [15]:

def extract_fact(sentence):
    doc = nlp(sentence)

    facts = []

    for token in doc:
        if token.dep_ == "ROOT":
            root = token
            subject = None
            dobj = None

            for child in root.children:
                if child.dep_ == "nsubj":
                    subject = child.text
                elif child.dep_ == "dobj":
                    dobj = child.text

            if subject and dobj:
                facts.append((subject, root.lemma_, dobj))

    return facts


# Example
sentence = "Lord Ju-ve built a cat  game."
facts = extract_fact(sentence)

print(facts)

[('Ju', 'build', 'game')]
